In [ ]:
# =============================================================================
# Experiment 10: Network Event Classification (Normal vs. Abnormal) using K-Means
# Objective: Use k-means to classify network events into normal and abnormal categories. [CO4]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, classification_report, silhouette_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Generate Synthetic Network Traffic Events Dataset
# Simulating Normal Network Events (high volume, tight distribution) and 
# Abnormal Network Events (low volume, anomalous packet/byte rates & error counts).
# =============================================================================

# Normal network events (450 samples)
X_normal, _ = make_blobs(
    n_samples=450,
    centers=[[120, 500, 480, 0.02, 1]],
    cluster_std=[[25, 80, 75, 0.01, 0.5]],
    random_state=42
)

# Abnormal / Intrusion network events (50 samples)
X_abnormal, _ = make_blobs(
    n_samples=50,
    centers=[[850, 4500, 150, 0.45, 12]],
    cluster_std=[[120, 600, 50, 0.12, 3.0]],
    random_state=42
)

X = np.vstack([X_normal, X_abnormal])
y_true = np.array([0] * 450 + [1] * 50)  # 0 = Normal, 1 = Abnormal (for evaluation only)

feature_names = ['Packet_Rate', 'Src_Bytes', 'Dst_Bytes', 'Error_Rate', 'Failed_Connections']
df = pd.DataFrame(X, columns=feature_names)
df['Actual_Category'] = np.where(y_true == 0, 'Normal', 'Abnormal')

# Shuffle dataset
shuffle_idx = np.random.RandomState(42).permutation(len(df))
df = df.iloc[shuffle_idx].reset_index(drop=True)
X = df[feature_names].values
y_true = (df['Actual_Category'] == 'Abnormal').astype(int).values

print(f"Total Network Events: {len(df)} | Normal: {(y_true==0).sum()} | Abnormal: {(y_true==1).sum()}")
print(df.head())

In [ ]:
# =============================================================================
# 3. Feature Scaling & K-Means Clustering (k = 2)
# =============================================================================

# Standardize features so high-magnitude features (Src_Bytes) don't dominate distances
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit K-Means with 2 clusters (Normal vs. Abnormal)
kmeans = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

# Map the smaller cluster to 'Abnormal' (1) and larger cluster to 'Normal' (0)
cluster_counts = np.bincount(cluster_labels)
abnormal_cluster = np.argmin(cluster_counts)
y_pred = (cluster_labels == abnormal_cluster).astype(int)

df['Predicted_Category'] = np.where(y_pred == 0, 'Normal', 'Abnormal')

sil_score = silhouette_score(X_scaled, cluster_labels)
print(f"K-Means Silhouette Score: {sil_score:.4f}")
print(f"Cluster Sizes -> Normal: {(y_pred==0).sum()}, Abnormal: {(y_pred==1).sum()}")

In [ ]:
# =============================================================================
# 4. Evaluate Clustering Against Ground Truth & Visualize Clusters (PCA)
# =============================================================================

print("--- Classification Report (K-Means Unsupervised Grouping) ---")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Abnormal'], yticklabels=['Normal', 'Abnormal'], ax=axes[0])
axes[0].set_title('K-Means Network Event Confusion Matrix')
axes[0].set_xlabel('Predicted Category')
axes[0].set_ylabel('Actual Category')

# Plot 2: 2D PCA Cluster Visualization with Centroids
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
centroids_pca = pca.transform(kmeans.cluster_centers_)

for label_val, name, color in [(0, 'Normal', 'steelblue'), (1, 'Abnormal', 'crimson')]:
    mask = (y_pred == label_val)
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, alpha=0.7, edgecolors='k', s=60)

axes[1].scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='gold', marker='X', s=250, edgecolors='black', label='Centroids')
axes[1].set_title('K-Means Clusters of Network Events (PCA Projection)')
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()